# SoundStream Demo

In [ ]:
from pathlib import Path
import subprocess
import os

from IPython.display import Audio, display

### Checking if demo is located in repo root. In other case cloning repo

In [ ]:
REPO_URL = "https://github.com/m-dumbduck/hse-ResearchDL-hw4.git"
REPO_FOLDER = "hse-ResearchDL-hw4"
EXPECTED_FILE = ".is_repo"
EXPECTED_TEXT_IN_FILE = "hse-ResearchDL-hw4"

def check_indicator(path):
    indicator = Path.cwd() / path / EXPECTED_FILE
    if not indicator.is_file():
        return False
    return indicator.read_text(encoding="utf-8").strip() == EXPECTED_TEXT_IN_FILE

In [ ]:
if not check_indicator('.') and not check_indicator(REPO_FOLDER):
    subprocess.run(["git", "clone", REPO_URL, REPO_FOLDER], check=True)
if check_indicator(REPO_FOLDER):
    os.chdir(REPO_FOLDER)

#### Installing dependencies

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch

#### Repo dependant imports

In [ ]:
from src.demo import load_mono_audio_from_url, prepare_audio, post_process_audio, resample_audio
from src.model.sound_stream import Generator

## SoundStream Codec demonstration

You can paste any audio link to `YOUR_AUDIO_LINK`. The default link is provided.

Note: `load_mono_audio_from_url` loads only first channel of audio, so the audio becomes mono.

In [ ]:
YOUR_AUDIO_LINK = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

In [ ]:
audio, sample_rate = load_mono_audio_from_url(YOUR_AUDIO_LINK)

### Your loaded audio

In [ ]:
display(Audio(data=audio[:, 0], rate=sample_rate))

We have our model trained only for sample rate $16$kHz, hence, we must resample your audio to used sample rate.

In [ ]:
MODEL_SAMPLE_RATE = 16000

In [ ]:
audio_16k = resample_audio(audio, sample_rate, MODEL_SAMPLE_RATE)

### Downloading and inferencing the model

Note: the model shrinks audio $200$ times, so `prepare_audio` paddes given audio to length divisible by $200$. `post_process_audio` cuts reconstructed audio back to the original length `length`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = Generator.from_pretrained("ldiujes/SoundStream").to(device).eval()

In [ ]:
audio_for_model, audio_length = prepare_audio(audio_16k, device)
with torch.no_grad():
    quantized_audio = model.encode_to_indexes(audio=audio_for_model)
    reconstructed_from_model = model.decode_from_indexes(indexes=quantized_audio)
reconstructed_audio_16k = post_process_audio(reconstructed_from_model, length=audio_length)

In [ ]:
quantized_audio.shape

### Results



#### Results at 16kHz sample rate (sample rate required for model)

Original audio resampled at $16000$

In [ ]:
display(Audio(data=audio_16k[:, 0], rate=MODEL_SAMPLE_RATE))

Reconstructed audio (sample rate $16000$ too)

In [ ]:
display(Audio(data=reconstructed_audio_16k[:, 0], rate=MODEL_SAMPLE_RATE))

#### We can also resampl reconstructed audio back to your audio sample rate

In [ ]:
reconstructed_audio_original_sr = resample_audio(reconstructed_audio_16k, sample_rate=MODEL_SAMPLE_RATE, target_sample_rate=sample_rate)

Your loaded audio

In [ ]:
display(Audio(data=audio[:, 0], rate=sample_rate))

Reconstructed audio in original sample rate

In [ ]:
display(Audio(data=reconstructed_audio_original_sr[:, 0], rate=sample_rate))